<a href="https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice

I selected Random Forest Regressor because my task is to predict a continuous Opportunity Score (baseline score), making this a regression problem. Random Forest can capture nonlinear relationships between search signals such as search volume, content age, average position, and CTR without requiring extensive preprocessing. It is also robust to different feature scales and provides feature importance, making it easier to interpret which signals contribute most to the predictions.

I will compare this model against my Week 4 baseline using the same target variable and evaluation metrics to determine whether the machine learning model improves prediction performance.

In [ ]:
!git clone https://github.com/HashamHassan-01/flyrank-ml-internship-hasham.git

Cloning into 'flyrank-ml-internship-hasham'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 168 (delta 70), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 1.89 MiB | 8.03 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
%cd flyrank-ml-internship-hasham

/content/flyrank-ml-internship-hasham


In [ ]:
import pandas as pd

df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
# Normalize signals
df["volume_score"] = df["search_volume"] / df["search_volume"].max()

df["age_score"] = df["content_age_days"] / df["content_age_days"].max()

df["rank_score"] = df["avg_position"] / df["avg_position"].max()

df["ctr_score"] = 1 - (df["ctr"] / df["ctr"].max())

# Create the target variable (same as Week 4)
df["baseline_score"] = (
    0.40 * df["volume_score"] +
    0.25 * df["rank_score"] +
    0.20 * df["age_score"] +
    0.15 * df["ctr_score"]
)

# Check that it was created
df[["baseline_score"]].head()

,baseline_score
0,0.226042
1,0.328927
2,0.237110
3,0.319830
4,0.287965


In [ ]:
# Remove rows that have missing values in the columns used for modeling
df = df.dropna(subset=[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr",
    "baseline_score"
])

print("Dataset shape after cleaning:", df.shape)

Dataset shape after cleaning: (27532, 49)


In [ ]:
# Features (input variables)
X = df[[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr"
]]

# Target variable
y = df["baseline_score"]

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (27532, 4)
Shape of y: (27532,)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used an 80/20 train-test split with random_state=42 to ensure reproducible results. The model is trained on 80% of the data and evaluated on the remaining 20%. This is the same validation design used throughout the experiment so the comparison with the baseline is fair.

With only 31 unique clients across 27,532 rows, each client contributes many pages to the dataset. A random 80/20 split means pages from the same client can appear in both the training and test sets, which risks the model partly learning client-specific patterns rather than generalizable ones. A more rigorous approach would be a group-based split (e.g., GroupShuffleSplit on client_id), ensuring no client appears in both sets. This wasn't implemented this week, but it's a known limitation of the current validation design.

In [ ]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    df["content_id"],
    test_size=0.20,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Target  :", y_train.shape)
print("Testing Target   :", y_test.shape)
print("Test IDs         :", id_test.shape)

Training Features: (22025, 4)
Testing Features : (5507, 4)
Training Target  : (22025,)
Testing Target   : (5507,)
Test IDs         : (5507,)


In [ ]:
print(df["client_id"].nunique(), "unique clients vs", df.shape[0], "rows")

31 unique clients vs 27532 rows


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained a Random Forest Regressor using the same input features and target variable as the baseline. The model was evaluated on the same test split using regression metrics (MAE, RMSE, and R²). The results are compared with the baseline to determine whether the machine learning model improves prediction accuracy.

In [ ]:
# Keep only rows that have values in all required columns
df = df.dropna(subset=[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr",
    "baseline_score"
])

print("New dataset shape:", df.shape)
print(df[[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr",
    "baseline_score"
]].isna().sum())

New dataset shape: (27532, 49)
search_volume       0
content_age_days    0
avg_position        0
ctr                 0
baseline_score      0
dtype: int64


In [ ]:
X = df[[
    "search_volume",
    "content_age_days",
    "avg_position",
    "ctr"
]]

y = df["baseline_score"]

print(X.shape)
print(y.shape)

(27532, 4)
(27532,)


In [ ]:
# Train the Random Forest model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Model trained successfully!")

# Show a few predictions
results = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": y_pred[:10]
})

results

Model trained successfully!


,Actual,Predicted
0,0.327807,0.327601
1,0.219829,0.219914
2,0.236921,0.237251
3,0.191055,0.190761
4,0.208947,0.208894
5,0.282368,0.282325
6,0.211554,0.210889
7,0.263013,0.264054
8,0.192479,0.192439
9,0.210352,0.210563


In [ ]:
# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", round(mae, 6))
print("Root Mean Squared Error (RMSE):", round(rmse, 6))
print("R² Score:", round(r2, 6))

Mean Absolute Error (MAE): 0.000626
Root Mean Squared Error (RMSE): 0.003581
R² Score: 0.995588


In [ ]:
# Compare baseline and Random Forest

# Random Forest ranking for the test set
rf_ranking = pd.DataFrame({
    "content_id": id_test.values,
    "rf_predicted_score": y_pred
}).sort_values("rf_predicted_score", ascending=False)

rf_top20 = set(rf_ranking.head(20)["content_id"])

# Week 4 baseline ranking for the SAME test rows
baseline_test = df[df["content_id"].isin(id_test)][["content_id", "baseline_score"]]
baseline_ranking = baseline_test.sort_values("baseline_score", ascending=False)

baseline_top20 = set(baseline_ranking.head(20)["content_id"])

# Overlap
overlap = rf_top20 & baseline_top20
print(f"Pages in both top-20 lists: {len(overlap)} out of 20")
print(f"Overlap %: {len(overlap)/20*100:.1f}%")

Pages in both top-20 lists: 15 out of 20
Overlap %: 75.0%


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest Regressor achieved a very high R² (0.9956) and very low MAE/RMSE. This is expected rather than a sign of unusually strong predictive power: baseline_score was created directly from the same four features used to train the model (search_volume, content_age_days, avg_position, ctr). So the model is essentially learning to reconstruct a known formula from its own inputs, rather than predicting an independent real-world outcome. The near-zero error mainly reflects that the relationship is deterministic, not that the model has discovered something meaningful in noisy data.

This also means MAE, RMSE, and R² can't be fairly reported for the Week 4 baseline. Week 4 didn't train a predictive model — it used a rule-based formula to score and rank pages, with no test-set predictions to measure error against. Because of this, a direct metric-for-metric comparison between Week 4 and Random Forest isn't possible on equal footing.

A fairer comparison would be to check how similarly the two approaches rank pages — for example, whether Random Forest and the Week 4 rule agree on which pages fall into the top 20 priority list. That comparison is model-agnostic and reflects what both approaches actually produce: a prioritized ranking, not a prediction with measurable error. When compared this way, Random Forest and the Week 4 baseline agreed on 15 of the top 20 priority pages (75% overlap), suggesting the two approaches are directionally consistent even though a direct error-metric comparison isn't possible.

This also means MAE, RMSE, and R² can't be fairly reported for the Week 4 baseline. Week 4 didn't train a predictive model — it used a rule-based formula to score and rank pages, with no test-set predictions to measure error against. Because of this, a direct metric-for-metric comparison between Week 4 and Random Forest isn't possible on equal footing.

A fairer comparison would be to check how similarly the two approaches rank pages — for example, whether Random Forest and the Week 4 rule agree on which pages fall into the top 20 priority list. That comparison is model-agnostic and reflects what both approaches actually produce: a prioritized ranking, not a prediction with measurable error.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Feature Importance

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance


,Feature,Importance
1,content_age_days,0.891992
2,avg_position,0.083634
0,search_volume,0.021204
3,ctr,0.003169


When compared this way, Random Forest and the Week 4 baseline agreed on 15 of the top 20 priority pages (75% overlap), suggesting the two approaches are directionally consistent even though a direct error-metric comparison isn't possible.

## Self-check

Before you submit, confirm each line honestly:

☑ Every section above is filled — markdown thinking AND the code that backs it.

☑ The notebook runs top to bottom with no errors (Runtime → Run all).

☑ No client names, URLs, or private queries anywhere.

☑ My claims use careful words: observed, measured, directional, decision-support.

☑ Committed to my repo under work/notebooks/ — then submit my repo URL on the card.